In [2]:
import pandas as pd

In [4]:
df = pd.read_csv("../data/cleaned_results.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")

In [5]:
print(df["home_team"].nunique())
print(df["away_team"].nunique())

327
321


In [6]:
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,result
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False,1872,draw
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False,1873,home_win
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False,1874,home_win
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False,1875,draw
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False,1876,home_win


In [7]:
all_teams = pd.concat([df["home_team"], df["away_team"]])

team_counts = all_teams.value_counts()

team_counts.head(20)

Sweden         1099
England        1088
Argentina      1066
Brazil         1058
Germany        1030
South Korea    1006
Hungary        1004
Mexico         1002
Uruguay         970
France          933
Italy           891
Poland          891
Switzerland     883
Netherlands     877
Denmark         872
Norway          870
Thailand        863
Austria         859
Belgium         851
Scotland        850
Name: count, dtype: int64

In [8]:
team_counts.describe()

count     336.000000
mean      293.339286
std       289.706858
min         1.000000
25%        26.750000
50%       228.500000
75%       486.750000
max      1099.000000
Name: count, dtype: float64

In [9]:
team_matches = {}

for team in set(df["home_team"]).union(set(df["away_team"])):

    matches = pd.concat([
        df[df["home_team"] == team],
        df[df["away_team"] == team]
    ])

    team_matches[team] = matches.sort_values("date")

In [10]:
team_matches["Argentina"].head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,result
145,1902-07-20,Uruguay,Argentina,0,6,Friendly,Montevideo,Uruguay,False,1902,away_win
155,1903-09-13,Argentina,Uruguay,2,3,Friendly,Buenos Aires,Argentina,False,1903,away_win
180,1905-08-15,Argentina,Uruguay,0,0,Copa Lipton,Buenos Aires,Argentina,False,1905,draw
193,1906-08-15,Uruguay,Argentina,0,2,Copa Lipton,Montevideo,Uruguay,False,1906,away_win
195,1906-10-21,Argentina,Uruguay,2,1,Copa Newton,Buenos Aires,Argentina,False,1906,home_win


In [11]:
def get_previous_matches(team, match_date, n=5):

    matches = team_matches[team]

    previous = matches[
        matches["date"] < match_date
    ]

    return previous.tail(n)

In [15]:
sample_match = df.iloc[30000]

team = sample_match["home_team"]

date = sample_match["date"]

get_previous_matches(team, date)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,result
29586,2005-10-12,Luxembourg,Estonia,0,2,FIFA World Cup qualification,Luxembourg,Luxembourg,False,2005,away_win
29617,2005-11-12,Finland,Estonia,2,2,Friendly,Helsinki,Finland,False,2005,draw
29651,2005-11-16,Poland,Estonia,3,1,Friendly,Ostrowiec Świętokrzyski,Poland,False,2005,home_win
29859,2006-03-01,Northern Ireland,Estonia,1,0,Friendly,Belfast,Northern Ireland,False,2006,home_win
29976,2006-05-28,Turkey,Estonia,1,1,Friendly,Hamburg,Germany,True,2006,draw


In [16]:
def calculate_form(team, match_date):

    previous = get_previous_matches(
        team,
        match_date,
        n=5
    )

    if len(previous) < 5:
        return None

    points = 0

    for _, row in previous.iterrows():

        if row["home_team"] == team:

            if row["home_score"] > row["away_score"]:
                points += 3

            elif row["home_score"] == row["away_score"]:
                points += 1

        else:

            if row["away_score"] > row["home_score"]:
                points += 3

            elif row["away_score"] == row["home_score"]:
                points += 1

    return points / 15

In [17]:
sample_match = df.iloc[30000]

calculate_form(
    sample_match["home_team"],
    sample_match["date"]
)

0.3333333333333333

In [18]:
feature_rows = []

In [19]:
for _, match in df.iterrows():

    home_team = match["home_team"]
    away_team = match["away_team"]
    match_date = match["date"]

    home_form = calculate_form(
        home_team,
        match_date
    )

    away_form = calculate_form(
        away_team,
        match_date
    )

    if home_form is None or away_form is None:
        continue

    feature_rows.append({
        "home_form": home_form,
        "away_form": away_form,
        "result": match["result"]
    })

In [20]:
features_df = pd.DataFrame(feature_rows)

features_df.head()

,home_form,away_form,result
0,0.333333,0.666667,away_win
1,0.866667,0.266667,home_win
2,0.266667,1.000000,home_win
3,0.800000,0.400000,home_win
4,0.000000,0.400000,away_win


In [21]:
features_df.shape

(47968, 3)

In [22]:
features_df["result"].value_counts()

result
home_win    23417
away_win    13522
draw        11029
Name: count, dtype: int64

In [23]:
features_df.to_csv(
    "../data/features_v1.csv",
    index=False
)

In [24]:
def avg_goal_difference(team, match_date):

    previous = get_previous_matches(
        team,
        match_date,
        n=5
    )

    if len(previous) < 5:
        return None

    goal_diffs = []

    for _, row in previous.iterrows():

        if row["home_team"] == team:

            gd = row["home_score"] - row["away_score"]

        else:

            gd = row["away_score"] - row["home_score"]

        goal_diffs.append(gd)

    return sum(goal_diffs) / len(goal_diffs)

In [ ]:
sample_match = df.iloc[30000]


In [ ]:
sample_match = df.iloc[30000]

avg_goal_difference(
    sample_match["home_team"],
    sample_match["date"]
)

-0.2

In [26]:
feature_rows = []

for _, match in df.iterrows():

    home_team = match["home_team"]
    away_team = match["away_team"]
    match_date = match["date"]

    home_form = calculate_form(
        home_team,
        match_date
    )

    away_form = calculate_form(
        away_team,
        match_date
    )

    home_gd = avg_goal_difference(
        home_team,
        match_date
    )

    away_gd = avg_goal_difference(
        away_team,
        match_date
    )

    if (
        home_form is None or
        away_form is None or
        home_gd is None or
        away_gd is None
    ):
        continue

    feature_rows.append({

        "home_form": home_form,
        "away_form": away_form,

        "home_goal_diff": home_gd,
        "away_goal_diff": away_gd,

        "result": match["result"]

    })

In [27]:
features_df = pd.DataFrame(feature_rows)

features_df.head()

,home_form,away_form,home_goal_diff,away_goal_diff,result
0,0.333333,0.666667,-0.4,1.2,away_win
1,0.866667,0.266667,2.2,-0.8,home_win
2,0.266667,1.000000,-1.8,4.4,home_win
3,0.800000,0.400000,3.6,-1.6,home_win
4,0.000000,0.400000,-3.8,-1.2,away_win


In [28]:
features_df.to_csv(
    "../data/features_v1.csv",
    index=False
)

In [29]:
features_df.shape

(47968, 5)

In [30]:
features_df.head()

,home_form,away_form,home_goal_diff,away_goal_diff,result
0,0.333333,0.666667,-0.4,1.2,away_win
1,0.866667,0.266667,2.2,-0.8,home_win
2,0.266667,1.000000,-1.8,4.4,home_win
3,0.800000,0.400000,3.6,-1.6,home_win
4,0.000000,0.400000,-3.8,-1.2,away_win


In [31]:
features_df["result"].value_counts()

result
home_win    23417
away_win    13522
draw        11029
Name: count, dtype: int64